# Modulo 4 · Lezione 1 — Introduzione ai database

**Corso di Intelligenza Artificiale — Livello base**

---

## Cosa vedremo oggi?

| # | Blocco | Cosa imparerai |
|---|--------|----------------|
| 1 | Cos'è un database? | Perché non bastano i file CSV |
| 2 | Il modello relazionale | Tabelle, righe, colonne |
| 3 | Chiavi primarie ed esterne | Il collante che collega le tabelle |
| 4 | SQL — il linguaggio | DDL vs DML |
| 5 | `CREATE TABLE` | Definire la struttura |
| 6 | `INSERT INTO` | Aggiungere dati |
| 7 | `UPDATE` e `DELETE` | Modificare ed eliminare |
| 8 | Tipi di dato in SQL | TEXT, INTEGER, REAL, BLOB, NULL |
| 9 | Pratica | Creare e popolare il database di un negozio |

> 💡 **Idea centrale:** Un database relazionale è una raccolta organizzata di tabelle collegate tra loro. SQL è il linguaggio universale per comunicare con esse.

---
## 1. Perché non bastano i file CSV?

```
PROBLEMA CON I CSV:

clientes.csv:
id,nombre,email,ciudad
1,Ana,ana@mail.com,Madrid
2,Carlos,carlos@mail.com,Barcelona

pedidos.csv:
id,cliente_nombre,cliente_email,producto,precio
101,Ana,ana@mail.com,Teclado,45.00
102,Ana,ana@mail.com,Ratón,25.00
103,Carlos,carlos@mail.com,Monitor,350.00

PROBLEMI:
❌ Duplicazione: nome ed email di Ana ripetuti in ogni ordine
❌ Incoerenza: se Ana cambia email, va aggiornata in due punti
❌ Nessuna integrità: un ordine può fare riferimento a un cliente inesistente
❌ Nessuna concorrenza: due persone che modificano lo stesso CSV insieme → corruzione
❌ Nessun indice: cercare un ordine richiede di scorrere l'intero file riga per riga
```

**Un database relazionale risolve tutti questi problemi:**
- Memorizza ogni dato una sola volta.
- Mantiene l'integrità (non puoi creare un ordine per un cliente che non esiste).
- Supporta più utenti simultanei.
- Ha indici per ricerche rapide su milioni di record.

---
## 2. Il modello relazionale

Il modello relazionale organizza i dati in **tabelle** (chiamate anche *relazioni*).

```
TABELLA: clientes
┌────┬──────────┬─────────────────┬───────────┐
│ id │ nombre   │ email           │ ciudad    │  ← colonne (attributi)
├────┼──────────┼─────────────────┼───────────┤
│  1 │ Ana      │ ana@mail.com    │ Madrid    │  ← riga (record / tupla)
│  2 │ Carlos   │ carlos@mail.com │ Barcelona │
│  3 │ Lucía    │ lucia@mail.com  │ Sevilla   │
└────┴──────────┴─────────────────┴───────────┘
  ↑
  chiave primaria
```

**Concetti fondamentali:**

| Concetto | Altri nomi | Descrizione |
|----------|-----------------|-------------|
| Tabella | Relazione, entità | Raccolta di dati dello stesso tipo |
| Riga | Record, tupla | Un singolo elemento della tabella |
| Colonna | Attributo, campo | Una proprietà di ogni elemento |
| Cella | Valore | Il dato concreto |
| Schema | Schema | La definizione della struttura (non i dati) |

---
## 3. Chiavi primarie ed esterne

### Chiave primaria (`PRIMARY KEY`)

Identifica **in modo univoco** ogni riga di una tabella.
- Non può mai essere `NULL`.
- Non può ripetersi.
- Di solito è un numero intero autoincrementale.

### Chiave esterna (`FOREIGN KEY`)

Una colonna che fa riferimento alla chiave primaria di un'altra tabella. È il meccanismo che **collega le tabelle** tra loro.

```
clientes                    pedidos
┌────┬────────┐             ┌─────┬─────────────┬─────────┐
│ id │ nombre │             │ id  │ cliente_id  │ total   │
├────┼────────┤             ├─────┼─────────────┼─────────┤
│  1 │ Ana    │◄────────────│ 101 │      1      │  45.00  │
│  2 │ Carlos │◄──────┐     │ 102 │      1      │  25.00  │
│  3 │ Lucía  │       └─────│ 103 │      2      │ 350.00  │
└────┴────────┘             └─────┴─────────────┴─────────┘
  ↑ PK                               ↑ FK
```

**Integrità referenziale:** la FK garantisce che `cliente_id = 5` non possa esistere in pedidos se non c'è nessun cliente con `id = 5`.

---
## 4. SQL — il linguaggio

SQL (*Structured Query Language*) si divide in due grandi categorie:

```
DDL — Data Definition Language    DML — Data Manipulation Language
──────────────────────────────    ────────────────────────────────
Definisce la STRUTTURA             Manipola i DATI

CREATE TABLE  → crea tabella       SELECT → interroga
ALTER TABLE   → modifica tabella   INSERT → inserisce
DROP TABLE    → elimina tabella    UPDATE → modifica
                                   DELETE → elimina
```

**SQLite** è il motore di database che useremo durante il modulo. È incluso in Python — non devi installare nulla. Salva l'intero database in un unico file `.db`.

```
MOTORI SQL più comuni:
SQLite   → file locale, perfetto per sviluppo e app piccole
PostgreSQL → open source, produzione, molto potente
MySQL    → molto diffuso nelle applicazioni web
SQL Server → ecosistema Microsoft
```

In [ ]:
import sqlite3
from pathlib import Path

# Connettersi (crea il file se non esiste)
DB_PATH = Path.home() / "tienda.db"
conn = sqlite3.connect(DB_PATH)

# Il cursore è l'oggetto che esegue le istruzioni SQL
cur = conn.cursor()

# Attivare le chiavi esterne in SQLite (sono disattivate di default)
cur.execute("PRAGMA foreign_keys = ON")

print(f"Connessione stabilita: {DB_PATH}")
print(f"Versione di SQLite: {sqlite3.sqlite_version}")

---
## 5. `CREATE TABLE` — Definire la struttura

```sql
CREATE TABLE nombre_tabla (
    columna1  TIPO  RESTRICCIONES,
    columna2  TIPO  RESTRICCIONES,
    ...
);
```

**Tipi di dato in SQLite:**

| Tipo | Uso | Esempio |
|------|-----|---------|
| `INTEGER` | Numeri interi | id, cantidad, año |
| `REAL` | Numeri decimali | precio, temperatura |
| `TEXT` | Testo | nombre, email, descripción |
| `BLOB` | Dati binari | immagini, file |
| `NULL` | Valore assente | (qualsiasi colonna può essere NULL) |

**Vincoli comuni:**

| Vincolo | Significato |
|-------------|-------------|
| `PRIMARY KEY` | Chiave primaria |
| `AUTOINCREMENT` | Si incrementa da solo |
| `NOT NULL` | Non può essere vuoto |
| `UNIQUE` | Senza duplicati |
| `DEFAULT valore` | Valore predefinito |
| `CHECK (condizione)` | Convalida un valore |
| `FOREIGN KEY` | Chiave esterna |

In [ ]:
# Eliminare le tabelle se esistono (per poter rieseguire la cella)
cur.executescript("""
    DROP TABLE IF EXISTS pedido_productos;
    DROP TABLE IF EXISTS pedidos;
    DROP TABLE IF EXISTS productos;
    DROP TABLE IF EXISTS clientes;
    DROP TABLE IF EXISTS categorias;
""")

# Tabella: categorias
cur.execute("""
    CREATE TABLE categorias (
        id        INTEGER PRIMARY KEY AUTOINCREMENT,
        nombre    TEXT    NOT NULL UNIQUE,
        descripcion TEXT
    )
""")

# Tabella: clientes
cur.execute("""
    CREATE TABLE clientes (
        id          INTEGER PRIMARY KEY AUTOINCREMENT,
        nombre      TEXT    NOT NULL,
        email       TEXT    NOT NULL UNIQUE,
        ciudad      TEXT    DEFAULT 'Desconocida',
        fecha_alta  TEXT    DEFAULT (date('now'))
    )
""")

# Tabella: productos
cur.execute("""
    CREATE TABLE productos (
        id           INTEGER PRIMARY KEY AUTOINCREMENT,
        nombre       TEXT    NOT NULL,
        precio       REAL    NOT NULL CHECK(precio >= 0),
        stock        INTEGER NOT NULL DEFAULT 0 CHECK(stock >= 0),
        categoria_id INTEGER NOT NULL,
        FOREIGN KEY (categoria_id) REFERENCES categorias(id)
    )
""")

# Tabella: pedidos
cur.execute("""
    CREATE TABLE pedidos (
        id          INTEGER PRIMARY KEY AUTOINCREMENT,
        cliente_id  INTEGER NOT NULL,
        fecha       TEXT    NOT NULL DEFAULT (date('now')),
        estado      TEXT    NOT NULL DEFAULT 'pendiente'
                            CHECK(estado IN ('pendiente','enviado','entregado','cancelado')),
        FOREIGN KEY (cliente_id) REFERENCES clientes(id)
    )
""")

# Tabella intermedia: pedido_productos (relazione molti a molti)
cur.execute("""
    CREATE TABLE pedido_productos (
        pedido_id   INTEGER NOT NULL,
        producto_id INTEGER NOT NULL,
        cantidad    INTEGER NOT NULL CHECK(cantidad > 0),
        precio_ud   REAL    NOT NULL,
        PRIMARY KEY (pedido_id, producto_id),
        FOREIGN KEY (pedido_id)   REFERENCES pedidos(id),
        FOREIGN KEY (producto_id) REFERENCES productos(id)
    )
""")

conn.commit()
print("Tabelle create correttamente")

# Verificare quali tabelle esistono
tablas = cur.execute("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name").fetchall()
print("Tabelle nel database:", [t[0] for t in tablas])

---
## 6. `INSERT INTO` — Aggiungere dati

```sql
-- Inserire una riga specificando le colonne
INSERT INTO tabla (col1, col2, col3)
VALUES (val1, val2, val3);

-- Inserire più righe insieme
INSERT INTO tabla (col1, col2)
VALUES (v1, v2),
       (v3, v4),
       (v5, v6);

-- Senza specificare le colonne (pericoloso: dipende dall'ordine)
INSERT INTO tabla VALUES (val1, val2, val3);
```

> Specifica sempre le colonne — rende il codice più chiaro e resistente a cambiamenti dello schema.

In [ ]:
# Inserire le categorie
cur.executemany(
    "INSERT INTO categorias (nombre, descripcion) VALUES (?, ?)",
    [
        ("Electrónica",  "Dispositivos y accesorios electrónicos"),
        ("Informática",  "Equipos y periféricos informáticos"),
        ("Hogar",        "Artículos para el hogar"),
        ("Oficina",      "Material de oficina y papelería"),
    ]
)

# Inserire i clienti
cur.executemany(
    "INSERT INTO clientes (nombre, email, ciudad) VALUES (?, ?, ?)",
    [
        ("Ana García",     "ana@mail.com",     "Madrid"),
        ("Carlos López",   "carlos@mail.com",  "Barcelona"),
        ("Lucía Martín",   "lucia@mail.com",   "Sevilla"),
        ("Pedro Sánchez",  "pedro@mail.com",   "Valencia"),
        ("Marta Jiménez",  "marta@mail.com",   "Madrid"),
    ]
)

# Inserire i prodotti
cur.executemany(
    "INSERT INTO productos (nombre, precio, stock, categoria_id) VALUES (?, ?, ?, ?)",
    [
        ("Smartphone X12",   599.99, 50, 1),
        ("Auriculares BT",    89.99, 120, 1),
        ("Cargador rápido",   29.99, 200, 1),
        ("Portátil Pro 15",  1199.00, 30, 2),
        ("Teclado mecánico",  85.00, 75, 2),
        ("Ratón inalámbrico", 45.00, 90, 2),
        ("Monitor 27\" 4K",   450.00, 25, 2),
        ("Lámpara escritorio", 35.00, 60, 3),
        ("Silla ergonómica",  299.00, 15, 3),
        ("Bolígrafos 10u",     4.99, 500, 4),
        ("Agenda 2025",        12.99, 150, 4),
        ("Post-it pack",        8.50, 300, 4),
    ]
)

# Inserire gli ordini
cur.executemany(
    "INSERT INTO pedidos (cliente_id, fecha, estado) VALUES (?, ?, ?)",
    [
        (1, "2025-01-10", "entregado"),
        (1, "2025-02-15", "enviado"),
        (2, "2025-01-20", "entregado"),
        (3, "2025-02-01", "entregado"),
        (3, "2025-03-05", "pendiente"),
        (4, "2025-03-10", "enviado"),
        (5, "2025-03-12", "pendiente"),
    ]
)

# Inserire le righe d'ordine
cur.executemany(
    "INSERT INTO pedido_productos (pedido_id, producto_id, cantidad, precio_ud) VALUES (?, ?, ?, ?)",
    [
        (1, 1, 1, 599.99),  # ordine 1: smartphone
        (1, 3, 2,  29.99),  # ordine 1: 2 caricabatterie
        (2, 2, 1,  89.99),  # ordine 2: auricolari
        (2, 5, 1,  85.00),  # ordine 2: tastiera
        (3, 4, 1, 1199.00), # ordine 3: portatile
        (3, 6, 1,  45.00),  # ordine 3: mouse
        (4, 8, 2,  35.00),  # ordine 4: 2 lampade
        (5, 9, 1, 299.00),  # ordine 5: sedia
        (5,10, 5,   4.99),  # ordine 5: 5 pacchi di penne
        (6, 7, 1, 450.00),  # ordine 6: monitor
        (7,11, 2,  12.99),  # ordine 7: 2 agende
        (7,12, 3,   8.50),  # ordine 7: 3 blocchetti post-it
    ]
)

conn.commit()
print("Dati inseriti")
print(f"  Categorie: {cur.execute('SELECT COUNT(*) FROM categorias').fetchone()[0]}")
print(f"  Clienti:   {cur.execute('SELECT COUNT(*) FROM clientes').fetchone()[0]}")
print(f"  Prodotti:  {cur.execute('SELECT COUNT(*) FROM productos').fetchone()[0]}")
print(f"  Ordini:    {cur.execute('SELECT COUNT(*) FROM pedidos').fetchone()[0]}")

---
## 7. `UPDATE` e `DELETE` — Modificare ed eliminare

```sql
-- Aggiornare righe
UPDATE tabla
SET columna1 = valor1,
    columna2 = valor2
WHERE condición;

-- Eliminare righe
DELETE FROM tabla
WHERE condición;
```

⚠️ **Includi SEMPRE `WHERE`** in `UPDATE` e `DELETE`. Senza di esso, l'operazione interessa **tutte** le righe della tabella.

```sql
UPDATE productos SET precio = 0;     -- ❌ azzera tutti i prezzi
DELETE FROM clientes;                -- ❌ elimina tutti i clienti

UPDATE productos SET precio = 0 WHERE id = 5;   -- ✅
DELETE FROM clientes WHERE id = 3;              -- ✅
```

In [ ]:
# UPDATE: aumentare il prezzo dello smartphone del 10%
cur.execute("SELECT nombre, precio FROM productos WHERE id = 1")
print("Prima dell'UPDATE:", cur.fetchone())

cur.execute("""
    UPDATE productos
    SET precio = ROUND(precio * 1.10, 2)
    WHERE id = 1
""")

cur.execute("SELECT nombre, precio FROM productos WHERE id = 1")
print("Dopo l'UPDATE:", cur.fetchone())

# UPDATE: cambiare la città di Ana
cur.execute("UPDATE clientes SET ciudad = 'Málaga' WHERE email = 'ana@mail.com'")
cur.execute("SELECT nombre, ciudad FROM clientes WHERE email = 'ana@mail.com'")
print("\nCittà di Ana:", cur.fetchone())

conn.commit()

In [ ]:
# Inserire un record di prova per poi eliminarlo
cur.execute("INSERT INTO clientes (nombre, email, ciudad) VALUES ('Prueba', 'prueba@test.com', 'Test')")
id_prueba = cur.lastrowid
print(f"Cliente di prova inserito con id={id_prueba}")

# Verificare
cur.execute("SELECT COUNT(*) FROM clientes")
print(f"Totale clienti prima del DELETE: {cur.fetchone()[0]}")

# DELETE
cur.execute("DELETE FROM clientes WHERE id = ?", (id_prueba,))
conn.commit()

cur.execute("SELECT COUNT(*) FROM clientes")
print(f"Totale clienti dopo il DELETE: {cur.fetchone()[0]}")

---
## Prima occhiata ai dati con `SELECT`

Non abbiamo ancora studiato `SELECT` in profondità — questo è argomento della Lezione 2. Ma lo usiamo già per vedere cosa abbiamo.

In [ ]:
import pandas as pd

def tabla(sql, con=conn):
    return pd.read_sql_query(sql, con)

print("=== CATEGORIE ===")
display(tabla("SELECT * FROM categorias"))

print("\n=== CLIENTI ===")
display(tabla("SELECT * FROM clientes"))

print("\n=== PRODOTTI ===")
display(tabla("SELECT * FROM productos ORDER BY categoria_id, nombre"))

---
## 9. Pratica — Creare e popolare un database

---

### Esercizio 1 — Creare uno schema

**Consegna:** Crea un database `biblioteca2.db` con il seguente schema:

- **`autores`**: id (PK), nombre (NOT NULL), nacionalidad, anio_nacimiento
- **`libros`**: id (PK), titulo (NOT NULL), isbn (UNIQUE), precio (≥ 0), paginas, autor_id (FK)
- **`socios`**: id (PK), nombre (NOT NULL), email (UNIQUE), fecha_alta (DEFAULT oggi)
- **`prestamos`**: id (PK), socio_id (FK), libro_id (FK), fecha_prestamo, fecha_devolucion (può essere NULL), devuelto (INTEGER DEFAULT 0)

Poi verifica che le quattro tabelle siano state create correttamente.

In [ ]:
# ✏️ IL TUO CODICE QUI
import sqlite3
from pathlib import Path


<details>
<summary>💡 Vedi soluzione — Esercizio 1</summary>

```python
import sqlite3
from pathlib import Path

conn2 = sqlite3.connect(Path.home() / "biblioteca2.db")
cur2 = conn2.cursor()
cur2.execute("PRAGMA foreign_keys = ON")

cur2.executescript("""
    DROP TABLE IF EXISTS prestamos;
    DROP TABLE IF EXISTS libros;
    DROP TABLE IF EXISTS socios;
    DROP TABLE IF EXISTS autores;

    CREATE TABLE autores (
        id              INTEGER PRIMARY KEY AUTOINCREMENT,
        nombre          TEXT    NOT NULL,
        nacionalidad    TEXT,
        anio_nacimiento INTEGER
    );

    CREATE TABLE libros (
        id        INTEGER PRIMARY KEY AUTOINCREMENT,
        titulo    TEXT    NOT NULL,
        isbn      TEXT    UNIQUE,
        precio    REAL    CHECK(precio >= 0),
        paginas   INTEGER,
        autor_id  INTEGER NOT NULL,
        FOREIGN KEY (autor_id) REFERENCES autores(id)
    );

    CREATE TABLE socios (
        id         INTEGER PRIMARY KEY AUTOINCREMENT,
        nombre     TEXT    NOT NULL,
        email      TEXT    NOT NULL UNIQUE,
        fecha_alta TEXT    DEFAULT (date('now'))
    );

    CREATE TABLE prestamos (
        id                INTEGER PRIMARY KEY AUTOINCREMENT,
        socio_id          INTEGER NOT NULL,
        libro_id          INTEGER NOT NULL,
        fecha_prestamo    TEXT    NOT NULL DEFAULT (date('now')),
        fecha_devolucion  TEXT,
        devuelto          INTEGER DEFAULT 0,
        FOREIGN KEY (socio_id) REFERENCES socios(id),
        FOREIGN KEY (libro_id) REFERENCES libros(id)
    );
""")
conn2.commit()

tablas = cur2.execute(
    "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name"
).fetchall()
print("Tabelle create:", [t[0] for t in tablas])
```
</details>

---
### Esercizio 2 — Popolare il database

**Consegna:** Inserisci dati in `biblioteca2.db`:

- Almeno quattro autori con nazionalità diverse.
- Almeno otto libri distribuiti tra gli autori.
- Almeno cinque soci.
- Almeno sei prestiti: alcuni restituiti, altri attivi.

Poi mostra il conteggio dei record in ogni tabella.

In [ ]:
# ✏️ IL TUO CODICE QUI


---
### Esercizio 3 — UPDATE a cascata

**Consegna:** Nel database `tienda.db`:

1. Applica uno sconto del 15% a tutti i prodotti della categoria «Oficina».
2. Cambia lo stato di tutti gli ordini «enviado» in «entregado».
3. Aumenta lo stock di tutti i prodotti con stock < 30 di 50 unità.
4. Verifica ogni modifica con una `SELECT`.

In [ ]:
# ✏️ IL TUO CODICE QUI
import pandas as pd


<details>
<summary>💡 Vedi soluzione — Esercizio 3</summary>

```python
import sqlite3, pandas as pd
from pathlib import Path

conn = sqlite3.connect(Path.home() / "tienda.db")
cur  = conn.cursor()

# 1. Sconto 15% su Oficina
cur.execute("""
    UPDATE productos
    SET precio = ROUND(precio * 0.85, 2)
    WHERE categoria_id = (SELECT id FROM categorias WHERE nombre = 'Oficina')
""")
print("Prodotti Oficina dopo lo sconto:")
display(pd.read_sql_query("""
    SELECT p.nombre, p.precio
    FROM productos p
    JOIN categorias c ON p.categoria_id = c.id
    WHERE c.nombre = 'Oficina'
""", conn))

# 2. Enviado → entregado
cur.execute("UPDATE pedidos SET estado = 'entregado' WHERE estado = 'enviado'")
print("\nStati degli ordini:")
display(pd.read_sql_query("SELECT estado, COUNT(*) AS total FROM pedidos GROUP BY estado", conn))

# 3. Stock basso → +50
cur.execute("UPDATE productos SET stock = stock + 50 WHERE stock < 30")
print("\nProdotti con stock precedentemente basso:")
display(pd.read_sql_query("SELECT nombre, stock FROM productos ORDER BY stock", conn))

conn.commit()
```
</details>

---
### Esercizio 4 — Integrità referenziale

**Consegna:** Verifica che i vincoli funzionino correttamente:

1. Prova a inserire un prodotto con `categoria_id = 99` (inesistente) — deve fallire.
2. Prova a inserire un cliente con un'email già esistente — deve fallire.
3. Prova a inserire un prodotto con `precio = -10` — deve fallire.
4. Cattura le eccezioni con `try/except` e mostra il messaggio di errore.

In [ ]:
# ✏️ IL TUO CODICE QUI


<details>
<summary>💡 Vedi soluzione — Esercizio 4</summary>

```python
import sqlite3
from pathlib import Path

conn = sqlite3.connect(Path.home() / "tienda.db")
cur  = conn.cursor()
cur.execute("PRAGMA foreign_keys = ON")

pruebas = [
    (
        "FK violata (categoria_id=99)",
        "INSERT INTO productos (nombre, precio, stock, categoria_id) VALUES ('X', 10, 1, 99)"
    ),
    (
        "UNIQUE violato (email duplicata)",
        "INSERT INTO clientes (nombre, email) VALUES ('Otro', 'ana@mail.com')"
    ),
    (
        "CHECK violato (prezzo negativo)",
        "INSERT INTO productos (nombre, precio, stock, categoria_id) VALUES ('Y', -5, 1, 1)"
    ),
]

for descripcion, sql in pruebas:
    try:
        cur.execute(sql)
        print(f"❌ {descripcion}: NON è stato generato alcun errore (qualcosa non va)")
    except sqlite3.IntegrityError as e:
        print(f"✅ {descripcion}")
        print(f"   Errore: {e}")
    conn.rollback()
```
</details>

---
### Esercizio 5 (sfida) — Funzione `poblar_bd`

**Consegna:** Scrivi la funzione Python `poblar_bd(conn, n_clientes, n_productos)` che generi dati fittizi e li inserisca:

- `n_clientes` clienti con nomi, email uniche e città casuali.
- `n_productos` prodotti con nomi, prezzi e stock casuali distribuiti tra le categorie esistenti.
- Un ordine per ogni cliente, ciascuno con tra due e quattro prodotti casuali.
- Restituisce un dict con i conteggi inseriti.

Provala con `poblar_bd(conn, 20, 30)`.

In [ ]:
# ✏️ IL TUO CODICE QUI
import sqlite3, random
from pathlib import Path


<details>
<summary>💡 Vedi soluzione — Esercizio 5</summary>

```python
import sqlite3, random
from pathlib import Path

NOMBRES   = ["Alba","Bea","Clara","Diana","Eva","Fátima","Gloria",
              "Héctor","Iván","Jaime","Kike","Luis","Marcos","Nacho",
              "Óscar","Pablo","Quique","Rosa","Sara","Tomás"]
CIUDADES  = ["Madrid","Barcelona","Valencia","Sevilla","Bilbao",
              "Zaragoza","Málaga","Murcia","Alicante","Valladolid"]
ADJETIVOS = ["Pro","Ultra","Lite","Plus","Max","Mini","Smart","Eco"]
TIPOS     = ["Kit","Pack","Set","Device","Tool","Gadget","System"]

def poblar_bd(conn, n_clientes=20, n_productos=30):
    cur = conn.cursor()

    # Ottenere gli ID delle categorie esistenti
    cat_ids = [r[0] for r in cur.execute("SELECT id FROM categorias").fetchall()]
    if not cat_ids:
        raise ValueError("Non ci sono categorie — crea prima le tabelle")

    # Inserire i clienti
    clientes_insertados = 0
    usados = set()
    intentos = 0
    while clientes_insertados < n_clientes and intentos < n_clientes * 3:
        intentos += 1
        nombre = random.choice(NOMBRES)
        tag = random.randint(100, 9999)
        email = f"{nombre.lower()}{tag}@demo.com"
        if email in usados: continue
        usados.add(email)
        ciudad = random.choice(CIUDADES)
        try:
            cur.execute("INSERT INTO clientes (nombre, email, ciudad) VALUES (?,?,?)",
                        (nombre, email, ciudad))
            clientes_insertados += 1
        except sqlite3.IntegrityError:
            pass

    # Inserire i prodotti
    for _ in range(n_productos):
        nombre = f"{random.choice(ADJETIVOS)} {random.choice(TIPOS)} {random.randint(1,99)}"
        precio = round(random.uniform(5, 999), 2)
        stock  = random.randint(0, 500)
        cat    = random.choice(cat_ids)
        cur.execute("INSERT INTO productos (nombre, precio, stock, categoria_id) VALUES (?,?,?,?)",
                    (nombre, precio, stock, cat))

    # Ordini
    prod_ids = [r[0] for r in cur.execute("SELECT id FROM productos").fetchall()]
    cli_ids  = [r[0] for r in cur.execute("SELECT id FROM clientes").fetchall()]
    for cid in cli_ids[-n_clientes:]:
        cur.execute("INSERT INTO pedidos (cliente_id) VALUES (?)", (cid,))
        pid = cur.lastrowid
        for prod in random.sample(prod_ids, min(random.randint(2, 4), len(prod_ids))):
            precio = cur.execute("SELECT precio FROM productos WHERE id=?", (prod,)).fetchone()[0]
            try:
                cur.execute("INSERT INTO pedido_productos VALUES (?,?,?,?)",
                            (pid, prod, random.randint(1, 3), precio))
            except sqlite3.IntegrityError:
                pass

    conn.commit()
    return {
        "clientes":  clientes_insertados,
        "productos": n_productos,
        "pedidos":   n_clientes,
    }


conn = sqlite3.connect(Path.home() / "tienda.db")
resultado = poblar_bd(conn, 20, 30)
print("Inseriti:", resultado)
```
</details>

---
## Riepilogo della lezione

```sql
-- ── DDL: definire la struttura ────────────────────────────────
CREATE TABLE tabla (
    id    INTEGER PRIMARY KEY AUTOINCREMENT,
    col   TEXT    NOT NULL UNIQUE,
    fk_id INTEGER REFERENCES otra_tabla(id)
);

DROP TABLE IF EXISTS tabla;

-- ── DML: inserire ──────────────────────────────────────────
INSERT INTO tabla (col1, col2) VALUES (v1, v2);

INSERT INTO tabla (col1, col2)
VALUES (v1, v2), (v3, v4);       -- più righe

-- ── DML: modificare ─────────────────────────────────────────
UPDATE tabla
SET col = valor
WHERE condición;                  -- SEMPRE con WHERE!

-- ── DML: eliminare ──────────────────────────────────────────
DELETE FROM tabla
WHERE condición;                  -- SEMPRE con WHERE!

-- ── Metainformazioni ────────────────────────────────────────
SELECT name FROM sqlite_master WHERE type='table';
PRAGMA foreign_keys = ON;
```

## Cosa vediamo nella prossima lezione?

Nella **Lezione 2** impariamo a **interrogare i dati** con `SELECT`, `WHERE`, `ORDER BY`, `LIMIT` e le funzioni di aggregazione (`COUNT`, `SUM`, `AVG`, `MIN`, `MAX`).

In [ ]:
conn.close()
print("Connessione chiusa")